In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Passo 1: Configuração do Ambiente (Célula 1)
Primeiro, precisamos instalar a biblioteca ultralytics e garantir que o ambiente reconheça as duas GPUs.

In [2]:
# Instala as dependências necessárias
# O -q deixa a instalação silenciosa para não poluir o log
!pip install -q ultralytics roboflow
!pip install "numpy<2.0" --force-reinstall
import torch
from ultralytics import YOLO

# Verificação de Hardware
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Número de GPUs detectadas: {gpu_count}")
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("ALERTA: GPUs não detectadas. Verifique as configurações do acelerador no Kaggle (lado direito da tela).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 106.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.3 MB/s eta 0:0

Passo 2: Baixar o Dataset do Roboflow (Célula 2)
Aqui você usará aquele código de exportação que mencionei anteriormente.

Vá no seu projeto do Roboflow > Versions > Export > Select YOLOv8.

Copie o código e substitua abaixo.

In [3]:
from roboflow import Roboflow

# --- SUBSTITUA ABAIXO PELA SUA CHAVE DO ROBOFLOW ---
# Cole aqui exatamente o código que o site te deu
# Exemplo:
# rf = Roboflow(api_key="SUA_KEY_AQUI")
# project = rf.workspace("seu-workspace").project("seu-projeto")
# dataset = project.version(1).download("yolov8")

!pip install roboflow

from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("roboflow_key")

rf = Roboflow(api_key=secret_value_0)
project = rf.workspace("orange-code-ai").project("6rainstorm-final-project-ircwn-iewca")
version = project.version(1)
dataset = version.download("yolov8")

# NOTA: O dataset será baixado na pasta atual (/kaggle/working/nome-do-projeto-versao)
# O comando abaixo ajuda a localizar o arquivo data.yaml automaticamente para evitar erro de caminho
import os
import glob

# Encontra o arquivo data.yaml baixado
try:
    data_yaml_path = glob.glob(os.path.join(os.getcwd(), "**", "data.yaml"), recursive=True)[0]
    print(f"Arquivo de configuração encontrado em: {data_yaml_path}")
except IndexError:
    print("ERRO: Dataset não encontrado. Verifique se você colou o código do Roboflow corretamente.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 6rainstorm-final-project-1 in yolov8:: 100%|██████████| 18884/18884 [00:02<00:00, 7639.98it/s] 


Arquivo de configuração encontrado em: /kaggle/working/6rainstorm-final-project-1/data.yaml


Passo 3: Treinamento Multi-GPU com Alta Resolução (Célula 3)
Este é o coração do script. Configurei para usar o YOLO11m com imgsz=1024 (para pegar os detalhes de sujeira) e distribuir a carga nas duas GPUs.

In [ ]:
# Inicializa o modelo Medium (YOLO11m)
# Ele vai baixar os pesos pré-treinados automaticamente na primeira execução
model = YOLO('yolo11m.pt')

# Configuração de Treinamento Otimizada para Kaggle 2x T4
# Se der erro de memória (CUDA OOM), reduza o 'batch' para 16 ou 12.
results = model.train(
    data=data_yaml_path,    # Caminho encontrado na célula anterior
    epochs=100,             # Recomendado para o modelo Medium aprender bem
    imgsz=960,             # Alta resolução para ver pequenos defeitos/sujeira
    batch=24,               # Batch total (será dividido: 12 imagens por GPU)
    device=[0, 1],          # USA AS DUAS GPUs (DDP)
    patience=20,            # Para se não melhorar após 20 épocas
    augment=True,           # Ativa data augmentation padrão
    name='yolo11m_solar_t4',# Nome do projeto para salvar os logs
    exist_ok=True,          # Sobrescreve se já existir a pasta
    verbose=True
)

Ultralytics 8.3.234 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/6rainstorm-final-project-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11m_solar_t4, nbs=64, nms=Fal

Passo 4: Validar e Zipar os Resultados (Célula 4)
No Kaggle, os arquivos somem se você fechar a sessão. Esta célula vai gerar um arquivo .zip com o seu modelo treinado (best.pt) e os gráficos de performance para você baixar para o seu computador.

In [ ]:
import shutil
from IPython.display import FileLink

# Define o caminho onde o treino foi salvo
# O YOLO salva em runs/detect/nome_do_projeto
output_dir = '/kaggle/working/runs/detect/yolo11m_solar_t4'

# Caminho do melhor peso
best_weight_path = f"{output_dir}/weights/best.pt"

print(f"Treino finalizado! Melhor modelo salvo em: {best_weight_path}")

# Zipar a pasta de resultados para download fácil
shutil.make_archive('meu_modelo_yolo11m', 'zip', output_dir)

print("\n--- DOWNLOAD ---")
print("Clique no link abaixo para baixar seu modelo e gráficos:")
display(FileLink(r'meu_modelo_yolo11m.zip'))